In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [21]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


In [22]:
from safetensors.torch import load_file

def get_lora_deltas(repo_name: str, lora_alpha: int = 32, r: int = 16) -> dict:
    """Compute ΔW = lora_B @ lora_A * (alpha/r) directly from adapter weights."""
    from huggingface_hub import hf_hub_download

    path = hf_hub_download(
        repo_id  = f"Srishtik/{repo_name}",
        filename = "adapter_model.safetensors"
    )
    adapter_weights = load_file(path)
    scale = lora_alpha / r

    # Group A and B matrices by layer
    layers = {}
    for key, val in adapter_weights.items():
        if "lora_A" in key:
            base_key = key.replace("lora_A.default.weight", "").replace("lora_A.weight", "")
            layers.setdefault(base_key, {})["A"] = val.float()
        elif "lora_B" in key:
            base_key = key.replace("lora_B.default.weight", "").replace("lora_B.weight", "")
            layers.setdefault(base_key, {})["B"] = val.float()

    # Compute ΔW for each layer
    deltas = {}
    for base_key, mats in layers.items():
        if "A" in mats and "B" in mats:
            deltas[base_key] = scale * (mats["B"] @ mats["A"])  # (d_out, d_in)

    return deltas


# ── Usage ──
reasoning_deltas = get_lora_deltas("3-adapter-merge-qwen-3-0.6B-reasoning-openmath-10k")
non_reasoning_deltas = get_lora_deltas("3-adapter-merge-qwen-3-0.6B-non-reasoning-finetome-10k")
ag_deltas=get_lora_deltas("3-adapter-merge-qwen-3-0.6B-ag-news-10k")
print(f"Adapter 1 layers: {len(reasoning_deltas)}")  
print(f"Adapter 2 layers: {len(non_reasoning_deltas)}")
print(f"Sample keys: {list(reasoning_deltas.keys())[:3]}")

Adapter 1 layers: 196
Adapter 2 layers: 196
Sample keys: ['base_model.model.model.layers.0.mlp.down_proj.', 'base_model.model.model.layers.0.mlp.gate_proj.', 'base_model.model.model.layers.0.mlp.up_proj.']


## Normalize model keys before merging

There is a mismatch between the keys of base model and the trained model. So we have to normalize them and then merge them or else it will show 0

In [23]:
def normalize_delta_keys(deltas: dict) -> dict:
    """
    Convert 'base_model.model.model.layers.0.self_attn.q_proj.' 
    to 'model.layers.0.self_attn.q_proj.weight'
    """
    normalized = {}
    for k, v in deltas.items():
        new_key = k.replace("base_model.model.", "")  # strip LoRA prefix
        new_key = new_key.rstrip(".")                  # remove trailing dot
        new_key = new_key + ".weight"                  # add back .weight suffix
        normalized[new_key] = v
    return normalized

reasoning_deltas     = normalize_delta_keys(reasoning_deltas)
non_reasoning_deltas = normalize_delta_keys(non_reasoning_deltas)
ag_deltas            = normalize_delta_keys(ag_deltas)

# Verify
print(f"Sample normalized key: {list(reasoning_deltas.keys())[:3]}")
print(f"Overlap with base_sd: {len(set(reasoning_deltas.keys()) & set(base_sd.keys()))}")  # should be ~63 (9 modules × 7 blocks for Qwen 0.6B, or whatever your actual count is)

Sample normalized key: ['model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight']
Overlap with base_sd: 196


In [24]:
deltas=[]
deltas.append(reasoning_deltas)
deltas.append(non_reasoning_deltas)
deltas.append(ag_deltas)

In [25]:
len(deltas)

3

In [26]:


from copy import deepcopy
import torch.nn.functional as F
from typing import Dict, Optional

def apply_delta_to_base(base_state_dict,merged_deltas):
    merged=deepcopy(base_state_dict)
    for key in merged_deltas:
        if key in merged:
            merged[key]=(base_state_dict[key].float()+merged_deltas[key]).to(base_state_dict[key].dtype)
    return merged



## Linear Merge

In [27]:
def linear_merge(deltas: list[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    n = len(deltas)
    weights = [1.0 / n] * n

    all_keys = set(deltas[0].keys())
    for d in deltas[1:]:
        all_keys &= set(d.keys())

    merged = {}
    for key in all_keys:
        merged[key] = torch.zeros_like(deltas[0][key].float())
        for weight, delta in zip(weights, deltas):
            merged[key] += weight * delta[key].float()

    return merged

## SVD Merge

In [28]:
def svd_merge(
    deltas: list[Dict[str, torch.Tensor]],
    weights: Optional[list[float]] = None,
    rank: Optional[int] = None,
) -> Dict[str, torch.Tensor]:
    """
    Linearly combine deltas, then SVD-truncate the result to `rank`.
    """
    n = len(deltas)
    if weights is None:
        weights = [1.0 / n] * n

    keys = set(deltas[0].keys())
    for d in deltas[1:]:
        keys &= set(d.keys())

    merged = {}
    for key in keys:
        combined = torch.zeros_like(deltas[0][key].float())
        for weight, delta in zip(weights, deltas):
            combined += weight * delta[key].float()

        if combined.dim() < 2:
            merged[key] = combined
            continue

        try:
            U, S, Vh = torch.linalg.svd(combined, full_matrices=False)
            r = rank if rank is not None else S.shape[0]
            r = min(r, S.shape[0])
            merged[key] = (U[:, :r] * S[:r].unsqueeze(0)) @ Vh[:r, :]
        except Exception:
            merged[key] = combined

    return merged

## TIES MERGE

In [29]:
def ties_merge(deltas:list[Dict[str,torch.Tensor]],weights:Optional[list[float]]=None,density:float=0.2)->Dict[str,torch.Tensor]:
    n=len(deltas)
    if weights is None:
        weights=[1.0/n]*n
    all_keys=set(deltas[0].keys())
    for d in deltas[1:]:
        all_keys&=set(d.keys())
    merged={}
    for key in all_keys:
        tensors=[d[key].float() for d in deltas]
        
        ## TRIM
        trimmed=[]
        for t in tensors:
            flat=t.abs().flatten()
            if flat.numel()==0:
                trimmed.append(t)
                continue
            k=max(1,int(density*flat.numel()))
            threshold=torch.topk(flat,k).values.min()
            mask=t.abs()>=threshold
            trimmed.append(t*mask)
            
    ## ELECT 
        sign_sum=sum(torch.sign(t) for t in trimmed)
        elected_sign=torch.sign(sign_sum)
        elected_sign[elected_sign==0]=1.0
    
    ## MERGE
        numerator=torch.zeros_like(tensors[0]) # Sum of accepted weighted updates
        denominator=torch.zeros_like(tensors[0]) # Total weight of accepted adapters

        for w, t in zip(weights,trimmed):
            agree_mask=(torch.sign(t)==elected_sign).float()
            numerator+=w*t*agree_mask
            denominator+=w*agree_mask
        denominator=torch.clamp(denominator,min=1e-6)
        merged[key]=numerator/denominator
    
    return merged

## DARE MERGE

In [30]:
def dare_merge(
    deltas: list[Dict[str, torch.Tensor]],
    weights: Optional[list[float]] = None,
    density: float = 0.2,
    seed: int = 42,
) -> Dict[str, torch.Tensor]:
    n=len(deltas)
    if weights is None:
        weights=[1.0/n]*n
    all_keys=set(deltas[0].keys())
    for d in deltas[1:]:
        all_keys&=set(d.keys())

    merged={}
    rng=torch.Generator()
    rng.manual_seed(seed)

    for key in all_keys:
        tensors=[d[key].float() for d in deltas]
        result=torch.zeros_like(tensors[0])

        for w,t in zip(weights,tensors):
            mask=torch.bernoulli(
                torch.full(t.shape,density),generator=rng
            ).to(t.device)
            dare_delta=t*mask/(density+1e-8)
            result+=w*dare_delta
        merged[key]=result
    return merged

 ## SLERP Merge

In [31]:
from typing import Union, Optional, Dict
import torch

def slerp_merge(
    deltas: list[Dict[str, torch.Tensor]],
    t: Optional[Union[float, list[float]]] = None,
    eps: float = 1e-8,
) -> Dict[str, torch.Tensor]:
    n = len(deltas)
    assert n >= 2, "SLERP merge expects at least 2 deltas"

    # Allow passing a single float when there are only 2 deltas
    if isinstance(t, float):
        assert n == 2, "Single float t only valid for exactly 2 deltas"
        t = [t]

    if t is None:
        t = [1.0 / (i + 2) for i in range(n - 1)]

    assert len(t) == n - 1, f"Expected {n-1} t values, got {len(t)}"

    def slerp_pair(d1, d2, t_val):
        merged = {}
        keys = set(d1.keys()) & set(d2.keys())
        for key in keys:
            v1 = d1[key].float().flatten()
            v2 = d2[key].float().flatten()
            original_shape = d1[key].shape
            n1 = torch.norm(v1)
            n2 = torch.norm(v2)
            if n1 < eps or n2 < eps:
                merged[key] = ((1 - t_val) * d1[key].float() + t_val * d2[key].float())
                continue
            v1_unit = v1 / n1
            v2_unit = v2 / n2
            dot = torch.clamp(torch.dot(v1_unit, v2_unit), -1.0 + eps, 1.0 - eps)
            omega = torch.acos(dot)
            sin_omega = torch.sin(omega)
            if sin_omega.abs() < eps:
                merged[key] = ((1 - t_val) * d1[key].float() + t_val * d2[key].float())
            else:
                coeff1 = torch.sin((1 - t_val) * omega) / sin_omega
                coeff2 = torch.sin(t_val * omega) / sin_omega
                interp_norm = (1 - t_val) * n1 + t_val * n2
                slerp_vec = (coeff1 * v1_unit + coeff2 * v2_unit) * interp_norm
                merged[key] = slerp_vec.reshape(original_shape)
        return merged

    result = deltas[0]
    for i in range(1, n):
        result = slerp_pair(result, deltas[i], t[i - 1])

    return result

In [32]:
def merge_adapters(
    method: str,
    base_state_dict: Dict[str, torch.Tensor],
    deltas: list[Dict[str, torch.Tensor]],
    weights: Optional[list[float]] = None,
    **kwargs,
) -> Dict[str, torch.Tensor]:
    """
    Unified entry point for all merge methods.

    Args:
        method:                one of ['linear', 'svd', 'ties', 'dare', 'dare_ties', 'slerp']
        base_state_dict:       base model weights
        finetuned_state_dicts: list of finetuned model state dicts (2 for most methods)
        weights:               per-model weights (default: uniform)
        **kwargs:              method-specific args (density, rank, t, seed, etc.)

    Returns:
        merged state dict (ready to load into model)
    """
    n = len(deltas)
    if weights is None:
        weights = [1.0 / n] * n

    

    if method == "linear":
        merged_delta = linear_merge(deltas)

    elif method == "svd":

        merged_delta = svd_merge(
            deltas,
            rank=kwargs.get("rank", None)
        )

    elif method == "ties":
        merged_delta = ties_merge(deltas, weights=weights, density=kwargs.get("density", 0.2))

    elif method == "dare":
        merged_delta = dare_merge(
            deltas, weights=weights,
            density=kwargs.get("density", 0.2),
            seed=kwargs.get("seed", 42)
        )

    elif method == "dare_ties":
        merged_delta = dare_ties_merge(
            deltas, weights=weights,
            density=kwargs.get("density", 0.2),
            seed=kwargs.get("seed", 42)
        )

    elif method == "slerp":
        n = len(deltas)
        if n == 2:
            t_default = 0.5
        else:
            t_default = [1.0 / (i + 2) for i in range(n - 1)]
        merged_delta = slerp_merge(deltas, t=kwargs.get("t", t_default))

    else:
        raise ValueError(f"Unknown method: {method}. Choose from: linear, svd, ties, dare, dare_ties, slerp")

    return apply_delta_to_base(base_state_dict, merged_delta)


In [33]:
def upload_merged_model(
    merged_sd: dict,
    method: str,
    tokenizer,
    hf_token: str,
    base_repo: str = "unsloth/Qwen3-0.6B",
    your_hf_username: str = "Srishtik",
    max_seq_length: int = 2048,
    dtype=torch.float16,
    push_to_hub: bool = True,
    save_local: bool = False,
    local_dir: str = "./merged_models",
):
    """
    Loads merged state dict into a fresh base model and uploads to HuggingFace.

    Args:
        merged_sd           : output of merge_adapters()
        method              : merge method name — used for repo naming
        tokenizer           : tokenizer from your training run
        hf_token            : your HuggingFace write token
        base_repo           : base model to load architecture from
        your_hf_username    : your HF username
        max_seq_length      : must match training config
        dtype               : float16 recommended for upload
        push_to_hub         : whether to push to HF Hub
        save_local          : whether to also save locally
        local_dir           : parent dir for local saves
    """
    import os
    from unsloth import FastLanguageModel

    repo_name = f"{your_hf_username}/Qwen3-0.6B-{method}-3-adapters-merged"
    print(f"[upload] Preparing model for method='{method}' → {repo_name}")

    # ── Load fresh base model to receive merged weights ──
    model, _ = FastLanguageModel.from_pretrained(
        model_name     = base_repo,
        max_seq_length = max_seq_length,
        load_in_4bit   = False,
        dtype          = dtype,
    )

    # ── Cast merged_sd to match model dtype before loading ──
    target_dtype = next(model.parameters()).dtype
    cast_sd = {
        k: v.to(target_dtype) if v.is_floating_point() else v
        for k, v in merged_sd.items()
    }

    # ── Load merged weights ──
    missing, unexpected = model.load_state_dict(cast_sd, strict=False)
    if missing:
        print(f"  [warn] Missing keys  : {len(missing)}  (e.g. {missing[:3]})")
    if unexpected:
        print(f"  [warn] Unexpected keys: {len(unexpected)} (e.g. {unexpected[:3]})")

    model.eval()

    # ── Save locally ──
    if save_local:
        save_path = os.path.join(local_dir, f"unsloth/Qwen3-0.6B-{method}-3-adapters-merged")
        os.makedirs(save_path, exist_ok=True)
        model.save_pretrained(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"  [local] Saved to {save_path}")

    # ── Push to HuggingFace Hub ──
    if push_to_hub:
        model.push_to_hub(repo_name, token=hf_token, private=False)
        tokenizer.push_to_hub(repo_name, token=hf_token, private=False)
        print(f"  [hub] Pushed → https://huggingface.co/{repo_name}")

    # ── Free memory ──
    del model, cast_sd
    torch.cuda.empty_cache()

    return repo_name

In [34]:
from transformers import AutoModelForCausalLM
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen3-0.6B",
    torch_dtype=torch.float16,
    device_map="cpu",
)
base_sd = {k: v.cpu() for k, v in base_model.state_dict().items()}
del base_model
torch.cuda.empty_cache()

print(f"Base keys: {len(base_sd)}")
print(f"Sample base keys: {list(base_sd.keys())[:3]}")


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Base keys: 311
Sample base keys: ['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.k_proj.weight']


In [35]:
from unsloth import FastLanguageModel
_,tokenizer=FastLanguageModel.from_pretrained(
    model_name= "unsloth/Qwen3-0.6B",
    max_seq_length=2048,
    load_in_4bit=False,
)

==((====))==  Unsloth 2026.6.7: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/Qwen3-0.6B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [36]:
HF_TOKEN =token ## Insert your own token
methods  = ["linear", "svd", "ties", "dare"]

for method in methods:
    
    merged_sd = merge_adapters(
        method = method,
        base_state_dict = base_sd,
        deltas = deltas,
        weights = [0.5, 0.5],
        density  = 0.2,   # ties / dare 
        rank = 16,    # svd
        t  = 0.5,   # slerp
        seed = 42,    # dare
    )

    upload_merged_model(
        merged_sd = merged_sd,
        method    = method,
        tokenizer = tokenizer,
        hf_token  = HF_TOKEN,
    )


[upload] Preparing model for method='linear' → Srishtik/Qwen3-0.6B-linear-3-adapters-merged
==((====))==  Unsloth 2026.6.7: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/Qwen3-0.6B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


README.md:   0%|          | 0.00/526 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpk7f6amxt/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


  [hub] Pushed → https://huggingface.co/Srishtik/Qwen3-0.6B-dare-3-adapters-merged


In [37]:
HF_TOKEN =token ## Insert your own token
methods = ["slerp"]
n = len(deltas)
t_value = 0.5 if n == 2 else [1.0 / (i + 2) for i in range(n - 1)]

for method in methods:
    merged_sd = merge_adapters(
        method          = method,
        base_state_dict = base_sd,
        deltas          = deltas,
        weights         = [1/3, 1/3, 1/3],
        density         = 0.2,
        rank            = 16,
        t               = t_value,   # ← now correctly [0.5, 0.5] for 3 deltas
        seed            = 42,
    )
    upload_merged_model(
        merged_sd = merged_sd,
        method    = method,
        tokenizer = tokenizer,
        hf_token  = HF_TOKEN,
    )


[upload] Preparing model for method='slerp' → Srishtik/Qwen3-0.6B-slerp-3-adapters-merged
==((====))==  Unsloth 2026.6.7: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/Qwen3-0.6B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpkdsgmk7b/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


  [hub] Pushed → https://huggingface.co/Srishtik/Qwen3-0.6B-slerp-3-adapters-merged
